# ⚡ PPSimplify — Google Colab Cloud Runner

Welcome to **PPSimplify**! This notebook runs the PPSimplify Web App on Google Colab. By running on Colab, you get:
1. **Free T4 GPU acceleration** for ultra-fast Whisper speech-to-text transcribing.
2. **Direct Browser Upload/Download** (no Google Drive mounting or configuration required).

---

### 🛠️ Step 1: Connect to GPU Runtime (Recommended)
Before running the cells, make sure you are using a GPU runtime:
1. Go to the top menu and select **Runtime** -> **Change runtime type**.
2. Under *Hardware accelerator*, select **T4 GPU** (or any other available GPU).
3. Click **Save**.

---

Let's run the cells below in order!

In [ ]:
# @title 📥 1. Clone Repository & Install Dependencies
# @markdown Run this cell to clone the PPSimplify code and install required packages (including PyTorch CUDA and Cloudflare tunneling).

import os

# Clone the repository if it doesn't exist, otherwise pull latest changes
REPO_DIR = '/content/PPS-Timestamp-AI-Tool-Upgrad-V2'
if not os.path.exists(REPO_DIR):
    print("Cloning PPSimplify repository...")
    !git clone https://github.com/AnshulUpgrad/PPS-Timestamp-AI-Tool-Upgrad-V2.git {REPO_DIR}
else:
    print("PPSimplify repository exists. Pulling latest updates...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

print("\nInstalling python packages (this may take 1-2 minutes)...")
!pip install -q -r requirements.txt
# Ensure CUDA GPU support is active for fast whisper inference
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("\nDownloading Cloudflare Tunnel client...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("\n✅ Setup completed successfully!")

In [ ]:
# @title 🚀 2. Start PPSimplify Server & Launch App
# @markdown Enter your Gemini API Key below, run this cell, and click the **Cloudflare Tunnel URL** that is generated to launch the web app.

GEMINI_API_KEY = "YOUR_API_KEY_HERE" # @param {type:"string"}
# @markdown *(You can leave it blank if you want to set it inside the web application settings or if you already configured a .env file).* 

import os
import subprocess
import time

# Set Gemini API Key in current environment
if GEMINI_API_KEY and GEMINI_API_KEY != "YOUR_API_KEY_HERE":
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    # Also write it to the .env file in the workspace
    with open(".env", "w") as f:
        f.write(f"GEMINI_API_KEY={GEMINI_API_KEY}\n")

# Start Flask in the background and log output to flask.log
print("Starting Flask server...")
with open("flask.log", "w") as log_file:
    flask_process = subprocess.Popen(["python", "app.py"], stdout=log_file, stderr=log_file)

# Wait for Flask to start and check health
time.sleep(3)

if flask_process.poll() is not None:
    print("❌ Flask failed to start! Here are the error logs:")
    with open("flask.log", "r") as f:
        print(f.read())
else:
    print("✅ Flask server is running successfully in the background.")
    print("\nCreating secure tunnel via Cloudflare...")
    print("Look for the 'https://...trycloudflare.com' link in the output below and click it to open PPSimplify:")
    print("--------------------------------------------------------------------------------------------------")
    # Start Cloudflare Tunnel
    !cloudflared tunnel --url http://localhost:5000